In [1]:
# cell 1 — imports and load data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
from pathlib import Path
from scipy import stats
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# style
sns.set_theme(style="darkgrid")

# connection
DB_PATH = Path("../data/database/steam.db")
conn = sqlite3.connect(DB_PATH)

# load data
games = pd.read_sql("SELECT * FROM games", conn)
genre_perf = pd.read_csv("../data/processed/genre_performance.csv")
price_tiers = pd.read_csv("../data/processed/price_tiers.csv")

# filter to games with meaningful ratings
games_filtered = games[games['total_ratings'] > 10].copy()

print(f"Total games: {len(games)}")
print(f"Games with >10 ratings: {len(games_filtered)}")

Total games: 27075
Games with >10 ratings: 20024


In [2]:
# cell 2 — Mann-Whitney U: free vs paid reach
free_games = games_filtered[games_filtered['price'] == 0]['owner_midpoint']
paid_games = games_filtered[games_filtered['price'] > 0]['owner_midpoint']

u_stat, p_value = stats.mannwhitneyu(free_games, paid_games, alternative='two-sided')

print(f"Free games: {len(free_games)} games, median owners: {free_games.median():,.0f}")
print(f"Paid games: {len(paid_games)} games, median owners: {paid_games.median():,.0f}")
print(f"\nMann-Whitney U statistic: {u_stat:,.0f}")
print(f"P-value: {p_value:.6f}")
print(f"\nResult: {'Statistically significant' if p_value < 0.05 else 'Not significant'} (α=0.05)")

Free games: 2272 games, median owners: 35,000
Paid games: 17752 games, median owners: 10,000

Mann-Whitney U statistic: 25,877,510
P-value: 0.000000

Result: Statistically significant (α=0.05)


In [3]:
# cell 3 — Kruskal-Wallis: satisfaction across genres
genre_satisfaction = pd.read_sql("""
    SELECT g.genre, gm.positive_ratio
    FROM game_genres g
    JOIN games gm ON g.appid = gm.appid
    WHERE gm.total_ratings > 10
    AND g.genre NOT IN (
    'Early Access', 'Free to Play', 'Indie',
    'Gore', 'Violent', 'Nudity', 'Sexual Content',
    'Animation & Modeling', 'Design & Illustration',
    'Utilities', 'Audio Production', 'Video Production',
    'Web Publishing', 'Education', 'Software Training',
    'Game Development', 'Photo Editing', 'Accounting'
    )
    AND gm.positive_ratio IS NOT NULL
""", conn)

# create a list of arrays, one per genre
groups = [
    group['positive_ratio'].values 
    for name, group in genre_satisfaction.groupby('genre')
]

h_stat, p_value = stats.kruskal(*groups)

print("=== Kruskal-Wallis Test ===")
print(f"H statistic: {h_stat:.2f}")
print(f"P-value: {p_value:.6f}")
print(f"Result: {'Statistically significant' if p_value < 0.05 else 'Not significant'} (α=0.05)")

print("\n=== Median satisfaction per genre ===")
print(genre_satisfaction.groupby('genre')['positive_ratio']
      .median().sort_values(ascending=False).round(3))

=== Kruskal-Wallis Test ===
H statistic: 493.47
P-value: 0.000000
Result: Statistically significant (α=0.05)

=== Median satisfaction per genre ===
genre
Adventure                0.769
Action                   0.763
RPG                      0.763
Casual                   0.759
Strategy                 0.734
Sports                   0.734
Simulation               0.714
Racing                   0.714
Massively Multiplayer    0.641
Name: positive_ratio, dtype: float64


In [5]:
# cell 4 — OLS Regression
from sklearn.preprocessing import LabelEncoder

# prepare features
reg_data = pd.read_sql("""
    SELECT 
        gm.positive_ratio,
        gm.price,
        gm.average_playtime,
        gm.achievements,
        gm.release_year,
        gm.owner_midpoint,
        g.genre
    FROM games gm
    JOIN game_genres g ON gm.appid = g.appid
    WHERE gm.total_ratings > 10
    AND gm.positive_ratio IS NOT NULL
    AND g.genre NOT IN (
        'Early Access', 'Free to Play', 'Indie',
        'Gore', 'Violent', 'Nudity', 'Sexual Content',
        'Animation & Modeling', 'Design & Illustration',
        'Utilities', 'Audio Production', 'Video Production',
        'Web Publishing', 'Education', 'Software Training',
        'Game Development', 'Photo Editing', 'Accounting'
    )
""", conn)

# create genre dummy variables
# drop_first=True avoids multicollinearity — Action becomes the baseline
genre_dummies = pd.get_dummies(reg_data['genre'], drop_first=True)

# build feature matrix
X = pd.concat([
    reg_data[['price', 'average_playtime', 'achievements', 'release_year']],
    genre_dummies
], axis=1)

# log transform skewed features
X['average_playtime'] = np.log1p(X['average_playtime'])
X['price'] = np.log1p(X['price'])

y = reg_data['positive_ratio']

# convert all columns to float
X = X.astype(float)

# add constant for intercept
X = sm.add_constant(X)
X.columns = X.columns.astype(str)

# fit model
model = sm.OLS(y, X).fit()

print(f"R-squared: {model.rsquared:.4f}")
print(f"Adjusted R-squared: {model.rsquared_adj:.4f}")
print(f"Number of observations: {model.nobs:.0f}")
print("\n=== COEFFICIENTS (sorted by absolute effect) ===")

results_df = pd.DataFrame({
    'coefficient': model.params,
    'p_value': model.pvalues,
    'significant': model.pvalues < 0.05
}).drop('const').sort_values('coefficient', ascending=False)

print(results_df.round(4).to_string())

R-squared: 0.0426
Adjusted R-squared: 0.0423
Number of observations: 37033

=== COEFFICIENTS (sorted by absolute effect) ===
                       coefficient  p_value  significant
price                       0.0312   0.0000         True
Casual                      0.0122   0.0001         True
Adventure                   0.0081   0.0060         True
RPG                         0.0068   0.0707        False
average_playtime            0.0046   0.0000         True
release_year                0.0042   0.0000         True
achievements               -0.0000   0.0000         True
Strategy                   -0.0192   0.0000         True
Sports                     -0.0287   0.0000         True
Racing                     -0.0394   0.0000         True
Simulation                 -0.0435   0.0000         True
Massively Multiplayer      -0.0774   0.0000         True
